# ETL_02b — Poblar `dim_variables_etf` (Energy / IPPU / Waste)
* **Proyecto:** INGEI AFOLU – DNCC / MADES – Paraguay  
* **Propósito:** Completar `inventario.dim_variables_etf` con los `variable_uid` de
Energy, IPPU y Waste
(FK `NOT NULL` a esta tabla).
* **Desarrollo:** Claudia Palacios | cpalacios01@gmail.com | +595994648273
* **No reemplaza a `ETL_02`** — las 32.694 filas de Agriculture + LULUCF ya cargadas no se tocan;
este notebook solo **agrega** las filas de los otros 3 sectores.


## 0. Configuración

In [ ]:
from pathlib import Path

BASE_DIR      = Path('.')
METADATA_PATH = BASE_DIR / 'etf-cli' / 'src' / 'unfccc' / 'etf' / 'assets' / 'metadata.json.lzma'
OUT_DIR       = BASE_DIR / 'out'
OUT_DIR.mkdir(exist_ok=True)

DB_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'dbname':   'ingei_afolu',
    'user':     'postgres',
    'password': 'postgres'
}
DB_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
)

METADATA_VER = '1.30.4'
BATCH_SIZE   = 1000

# ── Sectores a cargar en ESTE notebook (todo excepto AFOLU, ya cargado en ETL_02) ──
SECTORES_OTROS = {
    '3665c27e-d055-47d7-8393-5f934f3ced9d': 'energy',
    'fed65b84-cdad-4e38-8848-ea6af3c391bc': 'ippu',
    'b1e41219-79a2-493d-ba97-de0e4d7f9d0f': 'waste',
}

print(f'Metadata : {METADATA_PATH}')
print(f'Existe   : {METADATA_PATH.exists()}')

## 1. Imports

In [ ]:
import lzma
import json
import re
import time
import math
import uuid as uuid_lib
import pandas as pd
from sqlalchemy import create_engine, text, String

engine = create_engine(DB_URL)
print('✅ Imports OK — engine creado')

## 2. Cargar metadata ETF

In [ ]:
t0 = time.time()
with lzma.open(METADATA_PATH, 'rb') as f:
    meta = json.load(f)

root      = meta['Metadata'][0]
variables = root['variable']
ms = round((time.time() - t0) * 1000)

print(f'✅ Metadata cargada en {ms} ms — {len(variables):,} variables totales')

## 3. Filtrar variables de Energy / IPPU / Waste

In [ ]:
vars_otros = [v for v in variables if v.get('role_node_uid') in SECTORES_OTROS]

por_sector = {}
for v in vars_otros:
    s = SECTORES_OTROS[v['role_node_uid']]
    por_sector[s] = por_sector.get(s, 0) + 1

print(f'Variables Energy/IPPU/Waste total : {len(vars_otros):,}')
for s, n in por_sector.items():
    print(f'  {s:<10}: {n:,}')

## 4. Parsear

In [ ]:
BRACKET_PAT = re.compile(r'\[([^\]]+)\]')
CODE_PAT    = re.compile(
    r'^(\d+(?:\([A-Z]+\))?\.?[A-Z]?\d*(?:\.\d+)*(?:\.[a-z](?:\.[a-z])?(?:\.[a-z])?)?)'
)

GAS_NORM = {
    'CO\u2082': 'CO2', 'CH\u2084': 'CH4', 'N\u2082O': 'N2O',
    'Aggregate GHGs': 'GHG_AGG', 'no gas': None,
    'HFCs': 'HFC', 'PFCs': 'PFC', 'SF\u2086': 'SF6', 'NF\u2083': 'NF3',
}
GAS_SET = set(GAS_NORM.keys())

MEASURE_KEYWORDS = {
    'Emissions', 'Activity Data', 'Implied emission factor',
    'Emission factor information', 'Carbon stock change',
    'Indirect emissions', 'Annual change in stock', 'Area',
    'Population', 'Total Emissions', 'Source emissions',
    'Amount applied', 'Gains', 'Losses', 'Production',
    'Harvested area', 'Total area', 'Area burned', 'Biomass burned',
    'Nitrogen input', 'Total N excreted',
}

def parse_variable(v: dict, sector: str) -> dict:
    """Idéntica lógica a ETL_02 v2: nunca deja medida=None (NOT NULL)."""
    name    = v.get('name') or ''
    parts   = BRACKET_PAT.findall(name)
    cat_raw = parts[0] if parts else ''
    mc      = CODE_PAT.match(cat_raw)
    gas_raw = next((p for p in parts if p in GAS_SET), None)

    medida_fallback = False
    encontrada = next((p for p in parts if p in MEASURE_KEYWORDS), None)
    if encontrada is not None:
        medida = encontrada
    elif len(parts) > 2:
        medida = parts[2]
    elif len(parts) >= 1:
        medida = cat_raw
        medida_fallback = True
    else:
        medida = 'Sin clasificar'
        medida_fallback = True

    unidad    = parts[-1] if parts else None
    parametro = parts[-2] if len(parts) > 1 else None

    return {
        'variable_uid'     : v.get('uid'),
        'codigo_ipcc'      : mc.group(1) if mc else None,
        'nombre_completo'  : (name[:500] if name else f"[sin nombre] uid={v.get('uid')}"),
        'medida'           : medida,
        'gas_codigo'       : GAS_NORM.get(gas_raw) if gas_raw else None,
        'parametro'        : parametro,
        'unidad'           : unidad,
        'sector'           : sector,
        'es_calculada'     : v.get('is_calculated', False),
        'es_editable'      : v.get('is_editable', True),
        'nke_permitido'    : v.get('nke_allowed', True),
        'node_uid'         : v.get('node_uid'),
        'metadata_ver'     : METADATA_VER,
        '_medida_fallback' : medida_fallback,
    }

t0 = time.time()
records = [parse_variable(v, SECTORES_OTROS[v['role_node_uid']]) for v in vars_otros]
ms = round((time.time() - t0) * 1000)

df = pd.DataFrame(records)
n_fallback = int(df['_medida_fallback'].sum())

print(f'✅ Parseo completado en {ms} ms')
print(f'   Filas generadas        : {len(df):,}')
print(f'   Con código IPCC        : {df["codigo_ipcc"].notna().sum():,}')
print(f'   Con medida por fallback: {n_fallback:,}')
print()
print(df['sector'].value_counts().to_string())

## 5. Validación de integridad

In [ ]:
NOT_NULL_COLS = ['variable_uid', 'nombre_completo', 'medida', 'sector']

def es_uuid_valido(x):
    if x is None:
        return False
    try:
        uuid_lib.UUID(str(x))
        return True
    except (ValueError, TypeError, AttributeError):
        return False

problemas = []
for col in NOT_NULL_COLS:
    n_nulos = df[col].isna().sum()
    if n_nulos > 0:
        problemas.append(f'{n_nulos} filas con {col} nulo')

mask_uid_invalido = ~df['variable_uid'].apply(es_uuid_valido)
n_uid_invalido = int(mask_uid_invalido.sum())
if n_uid_invalido > 0:
    problemas.append(f'{n_uid_invalido} filas con variable_uid inválido')

mask_node_invalido = df['node_uid'].notna() & ~df['node_uid'].apply(es_uuid_valido)
n_node_invalido = int(mask_node_invalido.sum())

# variable_uid duplicado CONTRA LO YA CARGADO EN BD (Agriculture/LULUCF de ETL_02)
with engine.connect() as con:
    uids_existentes = set(
        r[0] for r in con.execute(text('SELECT variable_uid FROM inventario.dim_variables_etf'))
    )
mask_ya_en_bd = df['variable_uid'].apply(lambda x: str(x) in {str(u) for u in uids_existentes} if x else False)
n_ya_en_bd = int(mask_ya_en_bd.sum())
if n_ya_en_bd > 0:
    problemas.append(f'{n_ya_en_bd} variable_uid ya existen en BD (se excluyen para no duplicar)')

n_duplicados_internos = int(df['variable_uid'].duplicated().sum())
if n_duplicados_internos > 0:
    problemas.append(f'{n_duplicados_internos} variable_uid duplicados dentro del propio archivo')

print('── Validación de integridad ──')
for p in problemas:
    print(f'  ⚠️  {p}')
if not problemas:
    print('  ✅ Sin problemas detectados')

mask_cuarentena = (
    df['variable_uid'].isna() |
    mask_uid_invalido |
    mask_ya_en_bd |
    df['variable_uid'].duplicated(keep='first')
)
df_cuarentena = df[mask_cuarentena].copy()
df = df[~mask_cuarentena].copy()

print()
print(f'  Filas válidas para insertar : {len(df):,}')
print(f'  Filas en cuarentena         : {len(df_cuarentena):,}')

if len(df_cuarentena) > 0:
    cuarentena_path = OUT_DIR / 'dim_variables_etf_otros_sectores_cuarentena.csv'
    df_cuarentena.to_csv(cuarentena_path, index=False, encoding='utf-8-sig')
    print(f'  📄 Detalle guardado en: {cuarentena_path}')

## 6. Limpiar y normalizar UUIDs

In [ ]:
def limpiar_uuid(x):
    if x is None or (isinstance(x, str) and x.strip() == ''):
        return None
    try:
        return str(uuid_lib.UUID(str(x)))
    except (ValueError, TypeError, AttributeError):
        return None

antes_node_nulos = int(df['node_uid'].isna().sum())
df['variable_uid'] = df['variable_uid'].apply(limpiar_uuid)
df['node_uid']      = df['node_uid'].apply(limpiar_uuid)
despues_node_nulos = int(df['node_uid'].isna().sum())
node_normalizados_a_null = despues_node_nulos - antes_node_nulos

print(f'variable_uid nulos : {df["variable_uid"].isna().sum()}  (debe ser 0)')
print(f'node_uid nulos      : {despues_node_nulos:,}')
print('✅ Columnas UUID limpias')
assert df['variable_uid'].isna().sum() == 0

## 7. Insertar en `inventario.dim_variables_etf`

In [ ]:
DB_COLS = [
    'variable_uid','codigo_ipcc','nombre_completo','medida','gas_codigo',
    'parametro','unidad','sector','es_calculada','es_editable',
    'nke_permitido','node_uid','metadata_ver'
]
df_insert = df[DB_COLS].copy()

total   = len(df_insert)
batches = math.ceil(total / BATCH_SIZE)
t0      = time.time()
filas_con_error = []
filas_ok        = 0
batches_con_retry = 0

print(f'Insertando {total:,} filas en {batches} batches de {BATCH_SIZE}...')
print()

for i in range(batches):
    chunk = df_insert.iloc[i * BATCH_SIZE : (i + 1) * BATCH_SIZE].copy()
    try:
        chunk.to_sql(
            name='dim_variables_etf', schema='inventario', con=engine,
            if_exists='append', index=False, method='multi',
            dtype={'variable_uid': String, 'node_uid': String}
        )
        filas_ok += len(chunk)
    except Exception as e:
        batches_con_retry += 1
        print(f'  ⚠️  Batch {i} falló como bloque, reintentando fila por fila...')
        for idx, row in chunk.iterrows():
            fila = pd.DataFrame([row])
            try:
                fila.to_sql(
                    name='dim_variables_etf', schema='inventario', con=engine,
                    if_exists='append', index=False,
                    dtype={'variable_uid': String, 'node_uid': String}
                )
                filas_ok += 1
            except Exception as e2:
                filas_con_error.append({
                    'variable_uid': row.get('variable_uid'),
                    'nombre_completo': row.get('nombre_completo'),
                    'error': str(e2).splitlines()[0][:200]
                })
    if (i + 1) % 10 == 0 or (i + 1) == batches:
        print(f'  Batch {i+1:>3}/{batches} — {round((i+1)/batches*100):>3}% — {round(time.time()-t0,1)}s')

elapsed_total = round(time.time() - t0, 1)
print()
print(f'✅ Filas insertadas  : {filas_ok:,} / {total:,}')
if filas_con_error:
    print(f'❌ Filas con error   : {len(filas_con_error):,}')
    errores_path = OUT_DIR / 'dim_variables_etf_otros_sectores_errores.csv'
    pd.DataFrame(filas_con_error).to_csv(errores_path, index=False, encoding='utf-8-sig')
    print(f'   Detalle: {errores_path}')
print(f'   Tiempo total: {elapsed_total}s')

## 8. Verificación post-carga

In [ ]:
SQL_VERIFY = """
SELECT sector, COUNT(*) AS total_vars,
       COUNT(*) FILTER (WHERE codigo_ipcc IS NOT NULL) AS con_codigo_ipcc
FROM inventario.dim_variables_etf
GROUP BY sector
ORDER BY sector
"""
df_verify = pd.read_sql(SQL_VERIFY, engine)
print(df_verify.to_string(index=False))
print()
print(f'Total general en dim_variables_etf: {df_verify["total_vars"].sum():,}  (esperado: 71.612 al completar todos los sectores)')

## 9. Registrar en auditoría

In [ ]:
import json as json_lib

detalle = {
    'metadata_ver'             : METADATA_VER,
    'total_insertadas'         : int(filas_ok),
    'por_sector'                : {s: int(n) for s, n in df_verify.set_index('sector')['total_vars'].items()},
    'filas_en_cuarentena'      : int(len(df_cuarentena)),
    'filas_con_medida_fallback': int(n_fallback),
    'batches_con_retry'        : int(batches_con_retry),
    'filas_con_error_insercion': int(len(filas_con_error)),
    'tiempo_seg'               : elapsed_total,
}

with engine.begin() as con:
    con.execute(text("""
        INSERT INTO auditoria.log_pipeline (etapa, resultado, mensaje, detalle)
        VALUES ('ETL_02b_dim_variables_etf_otros_sectores', :resultado, :mensaje, CAST(:detalle AS jsonb))
    """), {
        'resultado': 'ok' if not filas_con_error else 'warning',
        'mensaje'  : f'dim_variables_etf ampliada con Energy/IPPU/Waste: {filas_ok:,} variables cargadas',
        'detalle'  : json_lib.dumps(detalle),
    })

print('✅ Registro de auditoría guardado')
print(json_lib.dumps(detalle, indent=3, ensure_ascii=False))

engine.dispose()
print('\nSiguiente paso → ETL_03_cargar_json_sectores.ipynb')